In [ ]:
!pip install open3d datasets plyfile huggingface_hub plotly wandb torch-geometric tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.3 MB/s eta 0:00:00


In [ ]:
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.2.0+cpu.html tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.1/774.1 kB 21.1 MB/s eta 0:00:00


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open3d as o3d
import numpy as np
import wandb
import random
import zipfile
from huggingface_hub import hf_hub_download
from torch_geometric.nn import GCNConv, knn_graph, global_max_pool
from torch_geometric.data import Data, Batch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_cluster/_version_cpu.so
  import torch_geometric.typing


Using device: cpu


In [ ]:
# import os
# from google.colab import userdata

# os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
# os.environ['HF_TOKEN'] = userdata.get('HUGGINGFACE')

# from huggingface_hub import hf_hub_download

# hf_hub_download(
#     repo_id="BGLab/AgriField3D",
#     filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
#     repo_type="dataset",
#     local_dir="/content/MaizeField3D"
# )

# import zipfile

# zip_path = "/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip"
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall("./MaizeField3D/extracted")

# print("Unzipped!")

import os

data_path = "/content/MaizeField3D/extracted"
ply_files = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".ply"):
            ply_files.append(os.path.join(root, file))

print(f"Total PLY files: {len(ply_files)}")

Total PLY files: 1045


In [ ]:
class PointCloudDataset(Dataset):
    def __init__(self, ply_files, num_points=2048):
        self.files = ply_files
        self.num_points = num_points

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        pcd = o3d.io.read_point_cloud(self.files[idx])
        points = np.asarray(pcd.points)

        # Point sampling
        if points.shape[0] > self.num_points:
            choice = np.random.choice(points.shape[0], self.num_points, replace=False)
            points = points[choice]
        elif points.shape[0] < self.num_points:
            choice = np.random.choice(points.shape[0], self.num_points, replace=True)
            points = points[choice]

        # Normalization (Center and Scale)
        center = points.mean(axis=0)
        points = points - center
        scale = np.max(np.linalg.norm(points, axis=1))
        if scale > 0:
            points = points / scale

        return torch.tensor(points, dtype=torch.float32)

In [ ]:
def knn_graph_custom(x, k, batch, loop=True):
    """
    Custom implementation of knn_graph that doesn't require torch-cluster.
    x: [N, 3]
    k: int
    batch: [N]
    """
    B = int(batch.max().item() + 1)
    edge_indices = []

    for b in range(B):
        mask = (batch == b)
        x_b = x[mask]
        indices = torch.where(mask)[0]

        # Compute pairwise distances
        dist = torch.cdist(x_b, x_b)

        # Get top k neighbors
        k_val = k if loop else k + 1
        _, col = dist.topk(k_val, largest=False)

        if not loop:
            col = col[:, 1:]

        row = torch.arange(x_b.size(0), device=x.device).view(-1, 1).repeat(1, col.size(1)).view(-1)
        col = col.reshape(-1)

        row = indices[row]
        col = indices[col]

        edge_indices.append(torch.stack([row, col], dim=0))

    return torch.cat(edge_indices, dim=1)

In [ ]:
class GCNAutoEncoder(nn.Module):
    def __init__(self, num_points=2048, latent_dim=128, k=20):
        super().__init__()
        self.k = k
        self.num_points = num_points

        # Encoder using GCNConv
        self.conv1 = GCNConv(3, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 128)
        self.conv4 = GCNConv(128, 256)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(128)
        self.bn4 = nn.BatchNorm1d(256)

        # Latent Projection
        self.fc1 = nn.Linear(64 + 64 + 128 + 256, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, num_points * 3)
        )

    def forward(self, x):
        B, N, _ = x.shape
        device = x.device

        # Reshape for PyTorch Geometric processing
        x_flat = x.view(-1, 3)
        batch = torch.arange(B, device=device).repeat_interleave(N)

        # Compute KNN graph edges using custom implementation
        edge_index = knn_graph_custom(x_flat, k=self.k, batch=batch, loop=True)

        # Encoder pass with LeakyReLU activations
        x1 = F.leaky_relu(self.bn1(self.conv1(x_flat, edge_index)), 0.2)
        x2 = F.leaky_relu(self.bn2(self.conv2(x1, edge_index)), 0.2)
        x3 = F.leaky_relu(self.bn3(self.conv3(x2, edge_index)), 0.2)
        x4 = F.leaky_relu(self.bn4(self.conv4(x3, edge_index)), 0.2)

        # Concatenate multi-scale features
        x_cat = torch.cat([x1, x2, x3, x4], dim=1)

        # Global feature extraction
        x_global = global_max_pool(x_cat, batch)

        # Latent representation
        z = self.fc1(x_global)

        # Reconstruction
        out = self.decoder(z)
        out = out.view(B, self.num_points, 3)

        return out

In [ ]:
def chamfer_distance(p1, p2):
    # Compute pairwise distances
    dist = torch.cdist(p1, p2)
    # Loss from p1 to p2 and p2 to p1
    loss1 = dist.min(dim=2)[0].mean()
    loss2 = dist.min(dim=1)[0].mean()
    return loss1 + loss2

In [ ]:
# Hyperparameters
num_points = 2048
latent_dim = 128
batch_size = 8
lr = 1e-3
epochs = 50

subset_files = ply_files[:500]

dataset = PointCloudDataset(subset_files, num_points=2048)

train_set, test_set = torch.utils.data.random_split(dataset, [400, 100])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = GCNAutoEncoder(latent_dim=128).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
print(torch.__version__)

2.10.0+cpu


In [ ]:
# Initialize W&B
# Note: Ensure you have run !wandb login or set WANDB_API_KEY environment variable
wandb.init(project="ppt_mesh_autoencoder", name="gcn_autoencoder_run")

In [ ]:
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for points in train_loader:
        points = points.to(device)
        recon = model(points)
        loss = chamfer_distance(points, recon)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    test_loss = 0
    with torch.no_grad():
        for points in test_loader:
            points = points.to(device)
            recon = model(points)
            loss = chamfer_distance(points, recon)
            test_loss += loss.item()

    test_loss /= len(test_loader)

    # Log to W&B
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "test_loss": test_loss
    })

    print(f"Epoch: {epoch:02d} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

wandb.finish()


Epoch: 00 | Train Loss: 0.126638 | Test Loss: 0.119835
Epoch: 01 | Train Loss: 0.106385 | Test Loss: 0.104103
Epoch: 02 | Train Loss: 0.102043 | Test Loss: 0.097968
Epoch: 03 | Train Loss: 0.098301 | Test Loss: 0.094933
Epoch: 04 | Train Loss: 0.097917 | Test Loss: 0.094488
Epoch: 05 | Train Loss: 0.092559 | Test Loss: 0.090138
Epoch: 06 | Train Loss: 0.090604 | Test Loss: 0.092334
Epoch: 07 | Train Loss: 0.090529 | Test Loss: 0.091789
Epoch: 08 | Train Loss: 0.089425 | Test Loss: 0.089742
Epoch: 09 | Train Loss: 0.088772 | Test Loss: 0.088149
Epoch: 10 | Train Loss: 0.086908 | Test Loss: 0.089351
Epoch: 11 | Train Loss: 0.085328 | Test Loss: 0.086668
Epoch: 12 | Train Loss: 0.085327 | Test Loss: 0.088103
Epoch: 13 | Train Loss: 0.084051 | Test Loss: 0.086136
Epoch: 14 | Train Loss: 0.082818 | Test Loss: 0.085262
Epoch: 15 | Train Loss: 0.082074 | Test Loss: 0.084890
Epoch: 16 | Train Loss: 0.081910 | Test Loss: 0.089457
Epoch: 17 | Train Loss: 0.082017 | Test Loss: 0.087326
Epoch: 18 

KeyboardInterrupt: 

In [ ]:
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "model_checkpoint.pth")

In [ ]:
checkpoint = torch.load("model_checkpoint.pth")

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

GCNAutoEncoder(
  (conv1): GCNConv(3, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 128)
  (conv4): GCNConv(128, 256)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=512, out_features=128, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=6144, bias=True)
  )
)

In [ ]:
def visualize_reconstruction(ply_files, model, device, num_points=2048):
    model.eval()
    file = random.choice(ply_files)
    pcd = o3d.io.read_point_cloud(file)
    points = np.asarray(pcd.points)

    # Sampling and Normalization
    idx = np.random.choice(points.shape[0], num_points, replace=False)
    points = points[idx]
    center = points.mean(axis=0)
    points = (points - center) / np.max(np.linalg.norm(points - center, axis=1))

    points_tensor = torch.tensor(points, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        recon = model(points_tensor)

    recon = recon.squeeze(0).cpu().numpy()

    fig = make_subplots(rows=1, cols=2,
                        specs=[[{"type":"scene"},{"type":"scene"}]],
                        subplot_titles=("Original", "GCN Reconstructed"))

    fig.add_trace(go.Scatter3d(x=points[:,0], y=points[:,1], z=points[:,2],
                               mode='markers', marker=dict(size=2, color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=recon[:,0], y=recon[:,1], z=recon[:,2],
                               mode='markers', marker=dict(size=2, color='red')), row=1, col=2)

    fig.update_layout(title_text=f"Reconstruction Result: {os.path.basename(file)}")
    fig.show()

visualize_reconstruction(subset_files, model, device)